# Regression - Pipelines and Data Leakage

In the previous lecture we standardized features before fitting Ridge and Lasso models. We used a naïve approach, calling `fit_transform` on the entire dataset. This is convenient but introduces a subtle problem called *data leakage* that can make our models appear better than they actually are.

This notebook introduces sklearn's `Pipeline`, which bundles preprocessing and modeling into a single end-to-end workflow. Pipelines are the backbone of practical ML in sklearn: they prevent data leakage, simplify code, and work seamlessly with cross-validation and hyperparameter tuning. We also cover when scaling is necessary and when it is not.

## Setup

Load the packages and configure environment.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

We continue with the Boston dataset from the previous two lectures.

In [ ]:
# download the data set directly from the web using pandas
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/Boston.csv"
boston = pd.read_csv(url)

X = boston.loc[:, 'zn':]
y = boston['crim']

## The Data Leakage Problem

In the regularization notebook, we scaled *all* the data before fitting:

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # fit + transform on ALL data
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_scaled, y)
```

The problem is that `fit_transform(X)` computes the mean and standard deviation from *every* observation, including the ones that should be held out for testing. When we later evaluate on a "test" set, the model has already seen information about those observations through the scaling parameters.

This is data leakage: information from outside the training set influences the model. The effect is usually small for scaling, but the principle matters. Leakage through more complex preprocessing can dramatically inflate performance estimates.

Leakage comes in several forms:

- *Train-test contamination*: preprocessing fitted on full data before splitting (the example above). Any step that learns from data - scaling, imputation, encoding, feature selection - can leak this way.
- *Target leakage*: features that encode information about the outcome that would not be available at prediction time. For example, using "account_closed_date" to predict whether a customer will churn.
- *Temporal leakage*: using future data to predict the past, common in time series problems where observations have a natural time ordering.

The first form is the most common in practice and the one we can prevent mechanically with Pipelines. The other forms require careful thinking about what information is available at prediction time.

### Demonstrating Leakage

To isolate the leakage effect, we need both approaches to use the same data in the same order. We shuffle the data to minimize the effect of any pre-existing order, e.g., sorted by name. Shuffling is done once at the start so that positional splits are representative, then compare scaling on all data (leaky) vs scaling on training data only (correct).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

rng = np.random.default_rng(42)

# rng.permutation returns a new array of shuffled indices
idx = rng.permutation(len(X))
X_shuffled = X.iloc[idx].reset_index(drop=True)
y_shuffled = y.iloc[idx].reset_index(drop=True)

split_point = int(len(X) * 0.7)

Similar to the leaky approach we used before. Fit the scaler (calculate mean and std) and transform (apply scaling) on all data, then split into train and test.

In [ ]:
scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(X_shuffled)

X_train_wrong = X_all_scaled[:split_point]
X_test_wrong = X_all_scaled[split_point:]
y_train = y_shuffled[:split_point]
y_test = y_shuffled[split_point:]

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_wrong, y_train)
print(f"Leaky approach\nTrain R²: {ridge.score(X_train_wrong, y_train):.4f}")
print(f"Test R²:  {ridge.score(X_test_wrong, y_test):.4f}")

To avoid leakage, split first, then get scaling parameters from *training data only* and use them to transform everything.

In [ ]:
X_train_raw = X_shuffled.iloc[:split_point]
X_test_raw = X_shuffled.iloc[split_point:]

scaler_correct = StandardScaler()
X_train_scaled = scaler_correct.fit_transform(X_train_raw)  # fit + transform on train
X_test_scaled = scaler_correct.transform(X_test_raw)          # transform only on test

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
print(f"Correct approach\nTrain R²: {ridge.score(X_train_scaled, y_train):.4f}")
print(f"Test R²:  {ridge.score(X_test_scaled, y_test):.4f}")

Same rows, same ordering - the only difference is where the scaler was fitted. The difference in R² is small because `StandardScaler` leaks only summary statistics (mean, std). But the principle matters: with more complex preprocessing steps the effect compounds.

In practice, we use `train_test_split` instead of manual shuffling and slicing. It handles shuffling, splitting, and index alignment in one call.

In [ ]:
# train_test_split shuffles by default - cleaner than manual shuffle + slice
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
print(f"With train_test_split\nTrain R²: {ridge.score(X_train_scaled, y_train):.4f}")
print(f"Test R²:  {ridge.score(X_test_scaled, y_test):.4f}")

The pattern to internalize:

- `fit_transform` on training data: learns *and* applies the transformation
- `transform` on test data: applies the *already learned* transformation without refitting

This two-step pattern applies to every preprocessing step:

- Imputation: if you fill missing values with the column mean computed from all data, each fold's test set already "knows" the mean of the full dataset, including itself
- Encoding: target encoding (replacing categories with the mean of the target) fitted on all data leaks target information into every observation
- Feature selection: if you select the top-k features by correlation with y using all data, the test set influenced which features were chosen

Getting it wrong in any one of them contaminates your evaluation.

### Why This Gets Worse with Cross-Validation

The manual approach above requires careful bookkeeping: fit the scaler on the training fold, transform both folds, repeat for each CV split. It is easy to make a mistake, and the code becomes verbose quickly. With multiple preprocessing steps the risk of error multiplies.

Let's try the naïve approach: scale everything first, then cross-validate.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# WRONG: scale first, then cross-validate
# The scaler has seen ALL the data, including each test fold
scaler_all = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X)

scores_leaky = cross_val_score(Ridge(alpha=1.0), X_scaled_all, y, cv=5, scoring='r2')
print(f"Leaky CV (no shuffle)")
print(f"Mean R²:  {scores_leaky.mean():.4f} (+/- {scores_leaky.std():.4f})")
print(f"Per-fold: {scores_leaky.round(4)}")

Large negative R² values on some folds - the model is performing far worse than simply predicting the mean. Two problems are colliding here.

First, the scaler was fit on all the data (leakage). Second, `cross_val_score(cv=5)` internally creates `KFold(n_splits=5, shuffle=False)`, which splits the data into five contiguous blocks *in their original row order*. If the data has any geographic, temporal, or other structure in its ordering, those blocks can be very different from each other. A model trained on four blocks may fail completely on the fifth because it has never seen anything like it.

The Boston dataset is ordered by town. Contiguous blocks group similar neighborhoods together, so some folds train on suburbs and test on urban areas (or vice versa). That is why the per-fold scores vary so wildly.

### Shuffle Defaults in sklearn

Why does `KFold` default to `shuffle=False`? It is the safest choice for ordered data. Time series, customer logs, and sensor readings have a natural sequence. Shuffling would let future observations leak into training folds, silently inflating scores. By defaulting to no shuffle, sklearn forces you to opt in rather than accidentally destroying temporal structure.

The general rule:

- *No shuffle* when observation order carries information (time series, sequential logs, longitudinal studies)
- *Shuffle* when row order is arbitrary or an artifact of data collection (our case - town ordering has no predictive meaning)

When shuffling, always set `random_state` for reproducibility.

In [ ]:
scores_leaky_shuffled = cross_val_score(
    Ridge(alpha=1.0), X_scaled_all, y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='r2'
)
print(f"Leaky CV (shuffled)")
print(f"Mean R²:  {scores_leaky_shuffled.mean():.4f} (+/- {scores_leaky_shuffled.std():.4f})")
print(f"Per-fold: {scores_leaky_shuffled.round(4)}")

Shuffling fixes the fold composition problem and R² is now positive. But this is still leaky - every fold's test set was influenced by the scaling parameters computed on the full dataset. For `StandardScaler`, the effect is too small to measure here: the mean and standard deviation computed from all 506 rows are nearly identical to those from the ~405 training rows. With more complex preprocessing - imputation, feature selection, target encoding - the gap between leaky and clean scores can be substantial.

The Pipeline solves this correctly regardless of how many preprocessing steps are involved.

## Pipelines

A `Pipeline` chains preprocessing and modeling into a single sklearn estimator. When you call `fit`, it fits each step in sequence, passing the transformed output of one step as input to the next. When you call `predict` or `score`, it transforms through each preprocessing step (without refitting) then predicts with the final estimator.

This provides several benefits:

- Leakage prevention: transformations learned on training data are automatically applied correctly to test data
- Code organization: the entire workflow is a single object with `fit`, `predict`, and `score` methods
- CV compatibility: `cross_val_score` handles the fit/transform split correctly for each fold
- Deployment: the whole workflow ships as one object

### Building a Pipeline

The simplest way to create a Pipeline is with `make_pipeline`, which names the steps automatically from the class names.

In [ ]:
from sklearn.pipeline import make_pipeline

# StandardScaler → Ridge in one object
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
pipe

The pipeline has two steps: `standardscaler` and `ridge`. You can see them listed in the output.

Fitting the pipeline fits both steps in sequence: the scaler learns the mean and std from the training data, transforms it, then Ridge fits on the scaled result.

In [ ]:
pipe.fit(X_train, y_train)

print(f"Train R²: {pipe.score(X_train, y_train):.4f}")
print(f"Test R²:  {pipe.score(X_test, y_test):.4f}")

When we call `pipe.score(X_test, y_test)`, the pipeline transforms `X_test` using the *already fitted* scaler (no refitting), then scores with Ridge. This is exactly the correct approach, without any manual bookkeeping.

The results match what we got with proper implementation of train-test-split, in 1/3rd the code (2 lines vs 6).

### Pipeline with Cross-Validation

The real payoff comes with cross-validation. Pass the pipeline to `cross_val_score` and it handles everything automatically. Here is what happens under the hood:

```text
Split X into n equal chunks
For each fold i = 1..n (n-1 chunks train, 1 chunk test):
  1. Fit StandardScaler on X_train_i          ← learns mean, std from train only
  2. Transform X_train_i using that scaler
  3. Fit Ridge on scaled X_train_i, y_train_i
  4. Transform X_test_i using the same scaler  ← no refit!
  5. Score Ridge on scaled X_test_i, y_test_i
  6. Record the score
Return array of n scores
```

The scaler is refit from scratch on each fold's training data. No information leaks from the test portion.

In [ ]:
# RIGHT: Pipeline handles scaling inside each CV fold
scores_clean = cross_val_score(
    pipe, X, y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42), scoring='r2'
)
print(f"Pipeline CV (shuffled)")
print(f"Mean R²:  {scores_clean.mean():.4f} (+/- {scores_clean.std():.4f})")

The scores match the leaky shuffled result above. For `StandardScaler` alone, the leakage effect is negligible - the scaling parameters barely change when computed on 506 vs ~405 rows. The value of the Pipeline here is not a different number; it is *guaranteed correctness*. You do not need to think about which step to fit where. As preprocessing grows more complex (imputation, encoding, feature selection), the Pipeline prevents leakage that manual bookkeeping would miss.

### Using `Pipeline` Directly

`make_pipeline` is convenient but gives auto-generated step names. For more control, use `Pipeline` directly with named steps.

In [ ]:
from sklearn.pipeline import Pipeline

pipe_named = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

pipe_named.fit(X_train, y_train)
print(f"Test R²: {pipe_named.score(X_test, y_test):.4f}")

Named steps are useful when you need to access individual components, for example to inspect the fitted model's coefficients.

In [ ]:
pipe_named.named_steps

In [ ]:
# access the Ridge model inside the pipeline
ridge_inside = pipe_named.named_steps['model']
print(f"Coefficients: {ridge_inside.coef_.round(3)}")
print(f"Intercept:    {ridge_inside.intercept_:.3f}")

These coefficients are in the *scaled* feature space (units of standard deviations), which makes them directly comparable: the largest absolute coefficient corresponds to the most influential feature after accounting for scale differences. This is one of the benefits of standardizing.

You rarely need to convert back to original units. Predictions already come out in the original y scale (the Pipeline only transforms X, not y). If you do need the scaler's parameters, they are accessible:

In [ ]:
# access the scaler to see what it learned
scaler_inside = pipe_named.named_steps['scaler']
print(f"Feature means: {scaler_inside.mean_.round(2)}")
print(f"Feature stds:  {scaler_inside.scale_.round(2)}")

### Pipeline with `RidgeCV`

Pipelines work with any sklearn estimator. Since `RidgeCV` selects the best alpha internally, wrapping it in a Pipeline gives us automatic scaling *and* automatic alpha selection.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

# Pipeline: scale → RidgeCV (LOO-CV for alpha selection)
pipe_ridgecv = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=np.logspace(-3, 3, 50))
)

pipe_ridgecv.fit(X_train, y_train)

ridge_result = pipe_ridgecv.named_steps['ridgecv']
print(f"Best alpha: {ridge_result.alpha_:.4f}")
print(f"Train R²:   {pipe_ridgecv.score(X_train, y_train):.4f}")
print(f"Test R²:    {pipe_ridgecv.score(X_test, y_test):.4f}")

In [ ]:
# Pipeline: scale → LassoCV (5-fold CV for alpha selection)
pipe_lassocv = make_pipeline(
    StandardScaler(),
    LassoCV(cv=5, random_state=42)
)

pipe_lassocv.fit(X_train, y_train)

lasso_result = pipe_lassocv.named_steps['lassocv']
print(f"Best alpha:    {lasso_result.alpha_:.4f}")
print(f"Train R²:      {pipe_lassocv.score(X_train, y_train):.4f}")
print(f"Test R²:       {pipe_lassocv.score(X_test, y_test):.4f}")
print(f"Features kept: {sum(lasso_result.coef_ != 0)} / {X.shape[1]}")

This is the recommended way to use regularized models: let the Pipeline handle scaling and let `RidgeCV`/`LassoCV` handle alpha selection. No manual steps, no leakage risk.

### Comparing Models with Pipelines

With Pipelines, comparing multiple models is clean and fair. Each pipeline encapsulates its own preprocessing, so we can evaluate them on the same data with the same CV strategy.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

# define pipelines for each model
models = {
    'Dummy (mean)': make_pipeline(DummyRegressor(strategy='mean')),
    'OLS':          make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (CV)':   make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))),
    'Lasso (CV)':   make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42)),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':<15} {'Mean R²':>8} {'Std':>8}")
print("-" * 33)

for name, pipe in models.items():
    scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    print(f"{name:<15} {scores.mean():>8.4f} {scores.std():>8.4f}")

The Dummy model scores near zero or slightly negative. R² measures how much better a model's predictions are compared to always predicting the test fold's mean. `DummyRegressor` predicts the *training* fold's mean, which differs slightly from the test fold's mean. That mismatch makes it marginally worse than the test fold's own baseline, pushing R² just below zero. This is arithmetic, not a modeling failure.

A negative R² from a real model, on the other hand, means it is performing *worse* than this trivial baseline - a sign that something has gone wrong (wrong features, missing preprocessing, or, as we saw earlier, unshuffled folds on structured data).

Notice that `DummyRegressor` does not need scaling (it just predicts the mean), but wrapping it in a pipeline is harmless and keeps the comparison code uniform.

This table is the beginning of the *baseline hierarchy* we'll develop throughout the course: Dummy → OLS → Ridge → Lasso → (next lecture: KNN). The gaps between scores reveal dataset characteristics. If OLS and Ridge are close, regularization is not adding much, meaning collinearity is not a major problem. If Lasso drops many features but maintains performance, the dataset has redundant predictors.

## When to Scale

Not every model needs standardized features. The general rule:

- Scale when the algorithm is sensitive to feature magnitudes: regularization (Ridge, Lasso, Elastic Net), distance-based methods (KNN, SVM), and gradient-based optimization (logistic regression, neural networks)
- Do not scale when the algorithm is invariant to feature magnitudes: decision trees and tree-based ensembles (Random Forest, Gradient Boosting) split on thresholds, so the absolute scale of features does not affect the result

In practice, scaling when it is not needed is harmless (the tree will find the same splits), while *not* scaling when it is needed can break the model (Ridge will penalize large-scale features disproportionately). When in doubt, scale.

The Pipeline makes this easy to manage: just include or exclude `StandardScaler` as a step.

## Summary

Four ideas from this lecture:

- Data leakage occurs when information from outside the training set influences the model. Fitting a scaler on all data before splitting is leakage.
- `KFold` defaults to `shuffle=False` to protect ordered data. For datasets without natural ordering, pass `KFold(shuffle=True, random_state=...)` to avoid unrepresentative folds.
- `Pipeline` bundles preprocessing and modeling so that `fit` learns from training data only and `transform`/`predict` apply that learned transformation. This prevents leakage automatically.
- Scale features for regularization and distance-based methods. Trees do not need scaling.

The pattern going forward: always use a Pipeline when your workflow includes preprocessing. It is safer, cleaner, and compatible with `cross_val_score`, `GridSearchCV`, and other sklearn tools we will use later.